# 06. Enterprise Multi-Agent System - Microsoft Agent Framework Edition

## 개요
엔터프라이즈급 멀티 에이전트 시스템을 구축합니다.

### 학습 목표
- 복잡한 워크플로우 오케스트레이션
- 상태 관리 및 체크포인팅
- 에러 핸들링 및 복구
- 모니터링 및 로깅
- 실제 비즈니스 시나리오 구현

## 1. 환경 설정

In [1]:
import os
import asyncio
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Annotated, Dict, Any, List
from pydantic import Field, BaseModel
from dotenv import load_dotenv

from agent_framework import ChatAgent, AgentThread
from agent_framework.azure import AzureOpenAIChatClient

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# 환경변수 로드
load_dotenv()

# Azure OpenAI Chat Client 생성
chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

# 작업 디렉토리
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

logger.info("✅ 환경 설정 완료")
print(f"   체크포인트 디렉토리: {CHECKPOINT_DIR.absolute()}")
print(f"   출력 디렉토리: {OUTPUT_DIR.absolute()}")

2026-01-26 19:32:23,913 - __main__ - INFO - ✅ 환경 설정 완료


   체크포인트 디렉토리: /home/euson/Azure-AI-Agent-MAF/checkpoints
   출력 디렉토리: /home/euson/Azure-AI-Agent-MAF/outputs


## 2. 데이터 모델 정의

In [2]:
# Pydantic 모델로 구조화된 데이터 정의

class CustomerRequest(BaseModel):
    """고객 요청"""
    request_id: str
    customer_name: str
    request_type: str  # "technical", "billing", "general"
    description: str
    priority: str  # "low", "medium", "high", "urgent"
    timestamp: str

class AnalysisResult(BaseModel):
    """분석 결과"""
    request_id: str
    category: str
    recommended_action: str
    estimated_time: str
    requires_escalation: bool

class Resolution(BaseModel):
    """해결 방안"""
    request_id: str
    solution: str
    steps: List[str]
    additional_resources: List[str]

class CustomerResponse(BaseModel):
    """고객 응답"""
    request_id: str
    customer_name: str
    response_text: str
    resolution_summary: str
    estimated_resolution_time: str
    status: str  # "resolved", "in_progress", "escalated"

logger.info("✅ 데이터 모델 정의 완료")

2026-01-26 19:32:23,935 - __main__ - INFO - ✅ 데이터 모델 정의 완료


## 3. 전문가 에이전트 팀 구성

In [3]:
# 접수 및 분류 에이전트
intake_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Customer Service Intake Specialist.
    
    Your role:
    1. Receive and understand customer requests
    2. Classify requests by type and priority
    3. Extract key information
    4. Determine urgency level
    
    Categories: technical, billing, general
    Priorities: low, medium, high, urgent
    """,
    name="IntakeAgent"
)

# 기술 지원 에이전트
technical_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Technical Support Specialist.
    
    Your role:
    1. Analyze technical issues
    2. Provide step-by-step solutions
    3. Suggest troubleshooting steps
    4. Recommend escalation if needed
    """,
    name="TechnicalAgent"
)

# 결제 지원 에이전트
billing_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Billing Support Specialist.
    
    Your role:
    1. Handle billing inquiries
    2. Explain charges and fees
    3. Process refund requests
    4. Update payment information
    """,
    name="BillingAgent"
)

# 일반 상담 에이전트
general_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a General Customer Service Representative.
    
    Your role:
    1. Handle general inquiries
    2. Provide product information
    3. Guide customers to appropriate resources
    4. Maintain friendly, professional tone
    """,
    name="GeneralAgent"
)

# 품질 관리 에이전트
qa_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Quality Assurance Specialist.
    
    Your role:
    1. Review proposed solutions
    2. Check for accuracy and completeness
    3. Ensure customer satisfaction
    4. Format final response professionally
    """,
    name="QAAgent"
)

logger.info("✅ 전문가 에이전트 팀 생성 완료")
print("   - IntakeAgent: 요청 접수 및 분류")
print("   - TechnicalAgent: 기술 지원")
print("   - BillingAgent: 결제 지원")
print("   - GeneralAgent: 일반 상담")
print("   - QAAgent: 품질 관리")

2026-01-26 19:32:23,968 - __main__ - INFO - ✅ 전문가 에이전트 팀 생성 완료


   - IntakeAgent: 요청 접수 및 분류
   - TechnicalAgent: 기술 지원
   - BillingAgent: 결제 지원
   - GeneralAgent: 일반 상담
   - QAAgent: 품질 관리


## 4. 고객 서비스 워크플로우 구축

In [4]:
class CustomerServiceWorkflow:
    """고객 서비스 워크플로우 관리"""
    
    def __init__(self):
        self.intake_agent = intake_agent
        self.technical_agent = technical_agent
        self.billing_agent = billing_agent
        self.general_agent = general_agent
        self.qa_agent = qa_agent
    
    async def intake_step(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """Step 1: 요청 접수 및 분류"""
        logger.info("📥 Step 1: Intake and Classification")
        
        classification = await self.intake_agent.run(
            f"""Classify this customer request:
            
            Customer: {request.get('customer_name')}
            Request: {request.get('description')}
            
            Provide:
            1. Type (technical/billing/general)
            2. Priority (low/medium/high/urgent)
            3. Key issues identified
            """
        )
        
        # Extract category from response
        classification_text = classification.text.lower()
        if "technical" in classification_text:
            category = "technical"
        elif "billing" in classification_text:
            category = "billing"
        else:
            category = "general"
        
        logger.info(f"   Classified as: {category}")
        
        return {
            "category": category,
            "classification_details": classification.text
        }
    
    async def route_by_category(self, request: Dict[str, Any], category: str, classification: str) -> Dict[str, Any]:
        """카테고리별 라우팅 및 처리"""
        logger.info(f"🔀 Routing to: {category}_support")
        
        if category == "technical":
            return await self.technical_support_step(request, classification)
        elif category == "billing":
            return await self.billing_support_step(request, classification)
        else:
            return await self.general_support_step(request, classification)
    
    async def technical_support_step(self, request: Dict[str, Any], classification: str) -> Dict[str, Any]:
        """기술 지원 처리"""
        logger.info("🔧 Step 2: Technical Support")
        
        solution = await self.technical_agent.run(
            f"""Provide technical support for:
            
            Classification: {classification}
            Issue: {request.get('description')}
            
            Provide:
            1. Detailed solution steps
            2. Expected resolution time
            3. Additional resources if needed
            """
        )
        
        return {"solution": solution.text}
    
    async def billing_support_step(self, request: Dict[str, Any], classification: str) -> Dict[str, Any]:
        """결제 지원 처리"""
        logger.info("💳 Step 2: Billing Support")
        
        solution = await self.billing_agent.run(
            f"""Handle billing inquiry:
            
            Classification: {classification}
            Issue: {request.get('description')}
            
            Provide:
            1. Explanation of charges
            2. Resolution steps
            3. Next actions
            """
        )
        
        return {"solution": solution.text}
    
    async def general_support_step(self, request: Dict[str, Any], classification: str) -> Dict[str, Any]:
        """일반 상담 처리"""
        logger.info("ℹ️ Step 2: General Support")
        
        solution = await self.general_agent.run(
            f"""Address general inquiry:
            
            Classification: {classification}
            Question: {request.get('description')}
            
            Provide:
            1. Clear answer
            2. Relevant information
            3. Additional resources
            """
        )
        
        return {"solution": solution.text}
    
    async def qa_review_step(self, request: Dict[str, Any], solution: str, category: str) -> Dict[str, Any]:
        """품질 검토 및 최종 응답 생성"""
        logger.info("✅ Step 3: Quality Assurance Review")
        
        final_response = await self.qa_agent.run(
            f"""Review and format final customer response:
            
            Customer: {request.get('customer_name')}
            Category: {category}
            Original Request: {request.get('description')}
            Proposed Solution: {solution}
            
            Create a professional, friendly response that:
            1. Acknowledges the customer's concern
            2. Provides clear solution steps
            3. Sets expectations
            4. Offers additional support
            """
        )
        
        return {
            "final_response": final_response.text,
            "status": "resolved"
        }
    
    async def process_request(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """고객 요청 처리"""
        logger.info(f"\n{'='*60}")
        logger.info(f"Processing Request: {request.get('customer_name')}")
        logger.info(f"{'='*60}")
        
        try:
            # Step 1: 접수 및 분류
            classification_result = await self.intake_step(request)
            category = classification_result["category"]
            classification = classification_result["classification_details"]
            
            # Step 2: 카테고리별 라우팅 및 처리
            support_result = await self.route_by_category(request, category, classification)
            solution = support_result["solution"]
            
            # Step 3: 품질 검토
            final_result = await self.qa_review_step(request, solution, category)
            
            # 결과 통합
            result = {
                "request_id": request.get("request_id"),
                "customer_name": request.get("customer_name"),
                "category": category,
                "classification": classification,
                "solution": solution,
                "final_response": final_result["final_response"],
                "status": final_result["status"]
            }
            
            logger.info(f"{'='*60}")
            logger.info("Request Processing Complete")
            logger.info(f"{'='*60}\n")
            
            return result
            
        except Exception as e:
            logger.error(f"Error processing request: {str(e)}")
            return {
                "request_id": request.get("request_id"),
                "status": "error",
                "error": str(e)
            }

# 워크플로우 인스턴스 생성
customer_service = CustomerServiceWorkflow()

logger.info("✅ 고객 서비스 워크플로우 생성 완료")

2026-01-26 19:32:24,008 - __main__ - INFO - ✅ 고객 서비스 워크플로우 생성 완료


## 5. 시나리오 테스트

In [5]:
# 테스트 시나리오
test_requests = [
    {
        "request_id": "REQ-001",
        "customer_name": "John Smith",
        "description": "My Azure OpenAI deployment is returning 429 errors. How can I fix this?",
        "timestamp": datetime.now().isoformat()
    },
    {
        "request_id": "REQ-002",
        "customer_name": "Sarah Johnson",
        "description": "I was charged twice for my subscription this month. Can you help?",
        "timestamp": datetime.now().isoformat()
    },
    {
        "request_id": "REQ-003",
        "customer_name": "Mike Chen",
        "description": "What are the differences between GPT-4 and GPT-4 Turbo?",
        "timestamp": datetime.now().isoformat()
    }
]

# 각 요청 처리
async def process_all_requests():
    results = []
    
    for request in test_requests:
        result = await customer_service.process_request(request)
        results.append({
            "request_id": request["request_id"],
            "customer": request["customer_name"],
            "category": result.get("category"),
            "response": result.get("final_response"),
            "status": result.get("status")
        })
        
        print(f"\n{'='*60}")
        print(f"Request ID: {request['request_id']}")
        print(f"Customer: {request['customer_name']}")
        print(f"Category: {result.get('category')}")
        print(f"\nFinal Response:\n{result.get('final_response')}")
        print(f"\nStatus: {result.get('status')}")
        print(f"{'='*60}\n")
    
    return results

results = await process_all_requests()

2026-01-26 19:32:24,036 - __main__ - INFO - 
2026-01-26 19:32:24,038 - __main__ - INFO - Processing Request: John Smith
2026-01-26 19:32:24,042 - __main__ - INFO - ============================================================
2026-01-26 19:32:24,044 - __main__ - INFO - 📥 Step 1: Intake and Classification
2026-01-26 19:32:25,396 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:25,408 - __main__ - INFO -    Classified as: technical
2026-01-26 19:32:25,408 - __main__ - INFO - 🔀 Routing to: technical_support
2026-01-26 19:32:25,409 - __main__ - INFO - 🔧 Step 2: Technical Support
2026-01-26 19:32:31,339 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:31,341 - __main__ - INFO - ✅ Step 3: Quality Assurance Review
2026-01-26 19:32:34,727 


Request ID: REQ-001
Customer: John Smith
Category: technical

Final Response:
Subject: Resolution Steps for Your Azure OpenAI 429 Error

Dear John Smith,

Thank you for reaching out regarding the 429 errors you're encountering with your Azure OpenAI deployment. We understand how disruptive this can be, and we're here to help you resolve the issue efficiently.

### Proposed Solution

After reviewing your situation, we have outlined a detailed guide to assist you in resolving the 429 error, which typically indicates that your request limit has been exceeded. Please follow the steps below:

1. **Review Quota and Limits**
   - Log into the [Azure portal](https://portal.azure.com).
   - Navigate to your **OpenAI resource**.
   - Go to the **Usage + Quotas** section to check your current request limits and ensure you remain within the limits of your subscription tier.

2. **Implement Backoff Strategy**
   - If you've hit the limits, we recommend implementing a **backoff strategy** in your a

2026-01-26 19:32:35,228 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:35,231 - __main__ - INFO -    Classified as: billing
2026-01-26 19:32:35,232 - __main__ - INFO - 🔀 Routing to: billing_support
2026-01-26 19:32:35,232 - __main__ - INFO - 💳 Step 2: Billing Support
2026-01-26 19:32:38,245 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:38,247 - __main__ - INFO - ✅ Step 3: Quality Assurance Review
2026-01-26 19:32:42,604 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:42,607 - __main__ - INFO - ============================================================
2026-01-26 19:32:42,607 - __main__ - INFO - 


Request ID: REQ-002
Customer: Sarah Johnson
Category: billing

Final Response:
Subject: Assistance with Your Recent Billing Inquiry

Dear Sarah Johnson,

Thank you for reaching out regarding the double charge for your subscription this month. I understand how concerning this issue can be, and I appreciate your patience as we work to resolve it.

### Explanation of Charges
Double charges may occur for several reasons, including:
- **Billing Errors:** A system glitch might have led to multiple charges.
- **Multiple Accounts:** If you have more than one account linked to the same payment method, charges for each account may appear.
- **Transaction Delays:** If a previous attempt to charge was unsuccessful, it might have been retried and resulted in an extra charge.

### Resolution Steps
To help resolve the double charge, please follow these steps:
1. **Verify Account Details:** Confirm that you have only one active account with us and that there are no subscriptions linked to multiple se

2026-01-26 19:32:43,132 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:43,134 - __main__ - INFO -    Classified as: technical
2026-01-26 19:32:43,135 - __main__ - INFO - 🔀 Routing to: technical_support
2026-01-26 19:32:43,136 - __main__ - INFO - 🔧 Step 2: Technical Support
2026-01-26 19:32:48,992 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:48,994 - __main__ - INFO - ✅ Step 3: Quality Assurance Review
2026-01-26 19:32:53,605 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:53,606 - __main__ - INFO - ============================================================
2026-01-26 19:32:53,607 - __main__ - I


Request ID: REQ-003
Customer: Mike Chen
Category: technical

Final Response:
Subject: Your Inquiry About GPT-4 vs. GPT-4 Turbo

Dear Mike Chen,

Thank you for reaching out with your question about the differences between GPT-4 and GPT-4 Turbo. We appreciate your interest in understanding these two models, and I'm here to help clarify.

### Key Differences Between GPT-4 and GPT-4 Turbo

1. **Architecture and Performance**: 
   - GPT-4 Turbo is an optimized version of GPT-4, designed for faster response times and reduced costs, while maintaining similar capabilities.

2. **Capabilities**: 
   - Both models are designed to understand and generate human-like text. However, GPT-4 Turbo typically delivers better efficiency, especially in applications requiring quick responses.

3. **Resource Utilization**:
   - GPT-4 Turbo utilizes fewer computational resources compared to GPT-4, making it ideal for tasks that need high throughput.

4. **Trade-offs**:
   - There may be minor differences in 

## 6. 성능 모니터링 및 분석

In [6]:
import time

class PerformanceMonitor:
    """워크플로우 성능 모니터링"""
    
    def __init__(self):
        self.metrics = []
    
    async def monitor_request(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """요청 처리 시간 측정"""
        start_time = time.time()
        
        result = await customer_service.process_request(request)
        
        elapsed_time = time.time() - start_time
        
        metric = {
            "request_id": request["request_id"],
            "category": result.get("category"),
            "processing_time": elapsed_time,
            "status": result.get("status")
        }
        
        self.metrics.append(metric)
        
        return result
    
    def get_summary(self) -> Dict[str, Any]:
        """성능 요약 통계"""
        if not self.metrics:
            return {}
        
        processing_times = [m["processing_time"] for m in self.metrics]
        
        return {
            "total_requests": len(self.metrics),
            "avg_processing_time": sum(processing_times) / len(processing_times),
            "min_processing_time": min(processing_times),
            "max_processing_time": max(processing_times),
            "by_category": self._group_by_category()
        }
    
    def _group_by_category(self) -> Dict[str, int]:
        """카테고리별 통계"""
        category_count = {}
        for metric in self.metrics:
            category = metric["category"]
            category_count[category] = category_count.get(category, 0) + 1
        return category_count

# 성능 모니터링 실행
monitor = PerformanceMonitor()

print("\n📊 Performance Monitoring...\n")

for request in test_requests:
    await monitor.monitor_request(request)

summary = monitor.get_summary()

print(f"\n{'='*60}")
print("PERFORMANCE SUMMARY")
print(f"{'='*60}")
print(f"Total Requests: {summary['total_requests']}")
print(f"Average Processing Time: {summary['avg_processing_time']:.2f}s")
print(f"Min Processing Time: {summary['min_processing_time']:.2f}s")
print(f"Max Processing Time: {summary['max_processing_time']:.2f}s")
print(f"\nBy Category:")
for category, count in summary['by_category'].items():
    print(f"  {category}: {count}")
print(f"{'='*60}\n")

2026-01-26 19:32:53,634 - __main__ - INFO - 
2026-01-26 19:32:53,636 - __main__ - INFO - Processing Request: John Smith
2026-01-26 19:32:53,638 - __main__ - INFO - ============================================================
2026-01-26 19:32:53,639 - __main__ - INFO - 📥 Step 1: Intake and Classification



📊 Performance Monitoring...



2026-01-26 19:32:54,226 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:32:54,228 - __main__ - INFO -    Classified as: technical
2026-01-26 19:32:54,229 - __main__ - INFO - 🔀 Routing to: technical_support
2026-01-26 19:32:54,230 - __main__ - INFO - 🔧 Step 2: Technical Support
2026-01-26 19:33:01,069 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:33:01,071 - __main__ - INFO - ✅ Step 3: Quality Assurance Review
2026-01-26 19:33:05,881 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:33:05,884 - __main__ - INFO - ============================================================
2026-01-26 19:33:05,886 - __main__ - I


PERFORMANCE SUMMARY
Total Requests: 3
Average Processing Time: 11.45s
Min Processing Time: 7.51s
Max Processing Time: 14.59s

By Category:
  technical: 2
  billing: 1



## 7. 에러 핸들링 및 복구

In [7]:
class ResilientWorkflow:
    """에러 복구 기능을 가진 워크플로우"""
    
    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries
        self.workflow = customer_service
    
    async def process_with_retry(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """재시도 로직을 포함한 요청 처리"""
        last_error = None
        
        for attempt in range(self.max_retries):
            try:
                logger.info(f"Attempt {attempt + 1}/{self.max_retries}")
                result = await self.workflow.process_request(request)
                logger.info("✅ Processing successful")
                return result
                
            except Exception as e:
                last_error = e
                logger.warning(f"❌ Attempt {attempt + 1} failed: {str(e)}")
                
                if attempt < self.max_retries - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    logger.info(f"⏳ Waiting {wait_time}s before retry...")
                    await asyncio.sleep(wait_time)
        
        # 모든 재시도 실패
        logger.error(f"❌ All retries failed. Last error: {str(last_error)}")
        return {
            "status": "failed",
            "error": str(last_error),
            "final_response": "We apologize, but we're experiencing technical difficulties. Please try again later."
        }

# 복원력 있는 워크플로우 테스트
resilient_workflow = ResilientWorkflow(max_retries=3)

test_request = {
    "request_id": "REQ-TEST",
    "customer_name": "Test User",
    "description": "This is a test request",
    "timestamp": datetime.now().isoformat()
}

result = await resilient_workflow.process_with_retry(test_request)
print(f"\nTest Result: {result.get('status')}")

2026-01-26 19:33:28,025 - __main__ - INFO - Attempt 1/3
2026-01-26 19:33:28,027 - __main__ - INFO - 
2026-01-26 19:33:28,029 - __main__ - INFO - Processing Request: Test User
2026-01-26 19:33:28,031 - __main__ - INFO - ============================================================
2026-01-26 19:33:28,034 - __main__ - INFO - 📥 Step 1: Intake and Classification
2026-01-26 19:33:28,541 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:33:28,543 - __main__ - INFO -    Classified as: general
2026-01-26 19:33:28,544 - __main__ - INFO - 🔀 Routing to: general_support
2026-01-26 19:33:28,545 - __main__ - INFO - ℹ️ Step 2: General Support
2026-01-26 19:33:30,375 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-01-26 19:33:30,378 - __main__ - INFO - ✅ Step 3:


Test Result: resolved


## 8. 보고서 생성

In [8]:
async def generate_report(results: List[Dict[str, Any]]):
    """처리 결과 요약 보고서 생성"""
    
    report_agent = ChatAgent(
        chat_client=chat_client,
        instructions="""
        You are a Business Analyst creating executive reports.
        
        Create comprehensive, data-driven reports with:
        1. Executive Summary
        2. Key Metrics
        3. Category Breakdown
        4. Recommendations
        """,
        name="ReportAgent"
    )
    
    # 데이터 요약
    summary_data = {
        "total_requests": len(results),
        "by_category": {},
        "resolution_rate": 0
    }
    
    for result in results:
        category = result.get("category", "unknown")
        summary_data["by_category"][category] = summary_data["by_category"].get(category, 0) + 1
        if result.get("status") == "resolved":
            summary_data["resolution_rate"] += 1
    
    summary_data["resolution_rate"] = (summary_data["resolution_rate"] / len(results)) * 100
    
    # 보고서 생성
    report = await report_agent.run(
        f"""Generate an executive report based on this customer service data:
        
        Total Requests Processed: {summary_data['total_requests']}
        Resolution Rate: {summary_data['resolution_rate']:.1f}%
        
        By Category:
        {json.dumps(summary_data['by_category'], indent=2)}
        
        Sample Requests:
        {json.dumps([{'id': r['request_id'], 'customer': r['customer'], 'category': r['category']} for r in results[:3]], indent=2)}
        
        Include:
        - Executive Summary
        - Performance Highlights
        - Areas for Improvement
        - Strategic Recommendations
        """
    )
    
    # 보고서 저장
    report_path = OUTPUT_DIR / f"customer_service_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    report_path.write_text(report.text)
    
    print(f"\n{'='*60}")
    print("EXECUTIVE REPORT")
    print(f"{'='*60}\n")
    print(report.text)
    print(f"\n{'='*60}")
    print(f"Report saved to: {report_path}")
    print(f"{'='*60}\n")

await generate_report(results)

2026-01-26 19:33:37,348 - httpx - INFO - HTTP Request: POST https://eusonopenai.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"



EXECUTIVE REPORT

# Executive Report: Customer Service Performance Analysis

## Executive Summary
This report provides an analysis of customer service request data for the period under review. In total, three customer service requests were processed, achieving a commendable resolution rate of 100%. The breakdown of the requests by category indicates a higher incidence of technical queries, reflecting a potential area for operational focus. This report also outlines key performance highlights, areas for improvement, and strategic recommendations for maintaining and enhancing customer service efficacy.

## Key Metrics
- **Total Requests Processed:** 3
- **Resolution Rate:** 100.0%
- **Requests by Category:**
  - **Technical:** 2 requests (66.7%)
  - **Billing:** 1 request (33.3%)

## Category Breakdown
### 1. Technical Requests (2)
- **Customer Requests:**
  - REQ-001: John Smith
  - REQ-003: Mike Chen
  
   The technical requests made up the majority of total requests, suggesting eithe

## 마무리

### 학습한 내용
1. ✅ 엔터프라이즈급 멀티 에이전트 시스템
2. ✅ 복잡한 워크플로우 오케스트레이션
3. ✅ 조건부 라우팅 및 동적 실행
4. ✅ 에러 핸들링 및 복구 메커니즘
5. ✅ 성능 모니터링 및 분석
6. ✅ 자동 보고서 생성
7. ✅ 구조화된 데이터 모델 (Pydantic)

### 엔터프라이즈 특징
- **확장성**: 수천 건의 요청 처리 가능
- **신뢰성**: 에러 복구 및 재시도 로직
- **관찰성**: 로깅 및 성능 메트릭
- **유지보수성**: 모듈화된 에이전트 및 워크플로우
- **타입 안정성**: Pydantic 모델 사용

### 실제 프로덕션 고려사항
1. **데이터베이스 통합**: 요청 및 응답 영구 저장
2. **큐 시스템**: 비동기 작업 처리 (RabbitMQ, Azure Service Bus)
3. **캐싱**: Redis 등을 활용한 응답 캐싱
4. **보안**: 인증, 권한 관리, 데이터 암호화
5. **스케일링**: 수평 확장, 로드 밸런싱
6. **모니터링**: Application Insights, Prometheus
7. **CI/CD**: 자동화된 배포 파이프라인

### Microsoft Agent Framework의 강점
- **엔터프라이즈 준비**: 프로덕션 환경을 위한 기능
- **타입 안정성**: 런타임 에러 방지
- **워크플로우**: 복잡한 비즈니스 로직 구조화
- **확장성**: Python과 .NET 양쪽 지원
- **표준 준수**: MCP, OpenTelemetry 등

### 다음 단계
- 실제 프로덕션 환경에 배포
- Azure AI Foundry와 통합
- 추가 도구 및 API 통합
- A/B 테스팅 및 최적화